# Self-healing pipeline - the assisted-triage workflow

This notebook walks the **whole self-healing loop** end to end so you can *watch* it work.
When the nightly `dbt build` fails, three failure-only steps run in CI:

| Stage | Script | AI? | What it does |
|-------|--------|-----|--------------|
| **1 - Capture** | `scripts/capture_failure.py` | no | Turns dbt's own artifacts into one clean `failure_context.json`. |
| **2 - Diagnose** | `scripts/diagnose_failure.py` | **one** Claude call | Structured diagnosis: likely cause, proposed fix, confidence, safety flags. |
| **3 - Propose** | `scripts/propose_fix.py` | no | Opens a human-approval GitHub issue (locally: writes the issue markdown). |

The key idea: **it proposes, a human approves.** The pipeline never edits models, never
writes to the warehouse, never opens a PR on its own.

Below we build the pipeline green, deliberately break a model, then let the triage layer
catch, explain, and propose a fix - and finally apply the fix and go green again.

> Run the cells top to bottom.

In [1]:
# --- Setup: run everything against the project, using this venv's dbt/python ---
import os, sys, json, pathlib, subprocess

PROJECT = pathlib.Path.home() / "Downloads" / "market-movers-dbt"
os.chdir(PROJECT)

PY  = sys.executable                                  # this venv's python
DBT = str(pathlib.Path(sys.executable).parent / "dbt")  # this venv's dbt

def run(cmd, tail=18):
    "Run a shell command in the project root; print the last `tail` lines + exit code."
    print(f"$ {cmd}\n")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    lines = (r.stdout + r.stderr).strip().splitlines()
    print("\n".join(lines[-tail:]))
    print(f"\n[exit {r.returncode}]")
    return r.returncode

print("project:", PROJECT)
print("dbt    :", DBT)

project: /Users/thiha.th/Downloads/market-movers-dbt
dbt    : /Users/thiha.th/Downloads/market-movers-dbt/.venv/bin/dbt


## Stage 0 - a healthy pipeline (green build)

We seed synthetic prices (offline, no network) and run `dbt build`, which runs every model
**and** its tests in one pass. This is the state CI expects each morning.

In [2]:
run(f"{PY} scripts/seed_sample.py")
rc = run(f"{DBT} build --profiles-dir .")
print("\nBUILD:", "GREEN ✅" if rc == 0 else "RED ❌")

$ /Users/thiha.th/Downloads/market-movers-dbt/.venv/bin/python scripts/seed_sample.py

Seeded 3600 synthetic rows; raw.prices now holds 3756 rows.

[exit 0]
$ /Users/thiha.th/Downloads/market-movers-dbt/.venv/bin/dbt build --profiles-dir .

13:51:22  26 of 33 OK created sql table model main.mart_sector_overview .................. [OK in 0.04s]
13:51:22  30 of 33 START test unique_mart_momentum_ticker ................................ [RUN]
13:51:22  27 of 33 PASS not_null_mart_portfolio_bias_ticker .............................. [PASS in 0.03s]
13:51:22  29 of 33 PASS not_null_mart_momentum_ticker .................................... [PASS in 0.03s]
13:51:22  28 of 33 PASS unique_mart_portfolio_bias_ticker ................................ [PASS in 0.03s]
13:51:22  31 of 33 START test not_null_mart_sector_overview_n_names ...................... [RUN]
13:51:22  32 of 33 START test not_null_mart_sector_overview_sector ....................... [RUN]
13:51:22  33 of 33 START test unique_mart_

## Stage 1 - something breaks overnight

To simulate a real failure, we introduce a typo into `mart_movers.sql`: the final `SELECT`
now references a column `close_prices` that doesn't exist (the real column is `close_price`).
This is the kind of change that slips in via a bad merge. We keep the original text so we can
restore it later.

In [5]:
model = PROJECT / "models" / "marts" / "mart_movers.sql"
ORIGINAL = model.read_text()                       # keep the good version to restore later

assert ORIGINAL.count("r.close_price,") == 1, "expected exactly one occurrence to break"
model.write_text(ORIGINAL.replace(
    "r.close_price,",
    "r.close_prices,  -- BUG: injected typo for the self-healing demo",
))
print("Injected a typo into mart_movers.sql (close_price -> close_prices).\n")

rc = run(f"{DBT} build --profiles-dir .")
print("\nBUILD:", "GREEN ✅" if rc == 0 else "RED ❌  (failure is intentional - triage takes over below)")

AssertionError: expected exactly one occurrence to break

## Stage 2a - Capture (no AI)

`capture_failure.py` reads dbt's own artifacts (`target/run_results.json` + `manifest.json`)
and writes one curated `failure_context.json`: which node failed, the error, and the exact
compiled SQL dbt ran. No model call here - just clean, machine-readable facts.

In [6]:
run(f"{PY} scripts/capture_failure.py")

ctx = json.loads((PROJECT / "failure_context.json").read_text())
print("\nstatus     :", ctx["status"])
print("n_failures :", ctx.get("n_failures"))
for f in ctx.get("failures", []):
    print(f"  - {f['status']:6} {f['resource_type']:6} {f['name']}")
    if f.get("message"):
        print(f"    message: {f['message'][:160]}")

$ /Users/thiha.th/Downloads/market-movers-dbt/.venv/bin/python scripts/capture_failure.py

capture_failure: status=failures_found failures=1 -> /Users/thiha.th/Downloads/market-movers-dbt/failure_context.json
  - error model mart_movers

[exit 0]

status     : failures_found
n_failures : 1
  - error  model  mart_movers
    message: Runtime Error in model mart_movers (models/marts/mart_movers.sql)
  Binder Error: Values list "r" does not have a column named "close_prices"
  
  LINE 32:     


## Stage 2b - Diagnose (exactly one Claude call)

`diagnose_failure.py` sends that curated context to Claude (`claude-opus-4-8`) and gets back a
**structured** diagnosis constrained to a JSON schema - so it's machine-checkable. Exactly one
call per failed run (a cost guard), and it flags anything that would touch an operating rule or
looks like an upstream data problem. If `ANTHROPIC_API_KEY` isn't set, it skips cleanly.

In [7]:
rc = run(f"{PY} scripts/diagnose_failure.py", tail=30)

diag_path = PROJECT / "diagnosis.json"
if diag_path.exists():
    print("\n" + "=" * 60)
    print(json.dumps(json.loads(diag_path.read_text()), indent=2))
else:
    print("\nNo diagnosis.json produced - the step skipped (no key / no network).")
    print("In CI this is harmless: the job still uploads the captured context.")

$ /Users/thiha.th/Downloads/market-movers-dbt/.venv/bin/python scripts/diagnose_failure.py

diagnose_failure: wrote /Users/thiha.th/Downloads/market-movers-dbt/diagnosis.json  (model=claude-opus-4-8)

{
  "failing_model": "model.market_movers.mart_movers",
  "likely_cause": "The mart_movers model references a column r.close_prices that does not exist in the returns CTE, which only defines close_price (singular). This is a typo/code bug in the model SQL (explicitly labeled as an injected typo), causing DuckDB's binder to fail resolving the column.",
  "proposed_fix": "Edit models/marts/mart_movers.sql to change r.close_prices to r.close_price in the final select list, then re-run with `dbt build --profiles-dir .` on a non-main branch to validate. No operating-rule-sensitive changes required.",
  "confidence": "high",
  "touches_operating_rules": false,
  "is_upstream_data_issue": false
}

-> confidence=high  flags: none
-> This is a PROPOSAL. A human reviews and approves before any chan

## Stage 3 - Propose (a human-approval issue)

`propose_fix.py` turns the diagnosis into a report a human acts on. In CI it opens a GitHub
issue (needs `issues: write`); locally it writes `proposed_fix_issue.md`. Below we render that
issue exactly as a human would see it - note the approval checklist and the "this is a proposal"
footer. **Nothing is applied automatically.**

In [8]:
run(f"{PY} scripts/propose_fix.py", tail=4)

issue = PROJECT / "proposed_fix_issue.md"
if issue.exists():
    from IPython.display import Markdown, display
    display(Markdown(issue.read_text()))
else:
    print("No proposal (no diagnosis was produced upstream).")

$ /Users/thiha.th/Downloads/market-movers-dbt/.venv/bin/python scripts/propose_fix.py

</details>

---
_This is a **proposal** produced by assisted triage. A human reviews and approves before any change. The pipeline never self-edits._

[exit 0]


# [assisted-triage] model.market_movers.mart_movers failed - proposed fix (high confidence)

## Assisted-triage: a fix is proposed for your approval

The nightly `dbt build` failed. `capture_failure.py` turned dbt's artifacts into a clean failure report, and `diagnose_failure.py` made **one** Claude API call for a structured diagnosis. This issue surfaces that diagnosis so a human can decide. **Nothing has been changed** - no models edited, no PR opened, no writes to the warehouse.

**Failing node(s):** `mart_movers`
**Diagnosis confidence:** 🟢 high

### Likely cause
The mart_movers model references a column r.close_prices that does not exist in the returns CTE, which only defines close_price (singular). This is a typo/code bug in the model SQL (explicitly labeled as an injected typo), causing DuckDB's binder to fail resolving the column.

### Proposed fix (for a human to apply)
Edit models/marts/mart_movers.sql to change r.close_prices to r.close_price in the final select list, then re-run with `dbt build --profiles-dir .` on a non-main branch to validate. No operating-rule-sensitive changes required.

### Safety flags
- None raised.

### Human approval checklist
- [ ] Read the proposed fix and confirm it addresses the real cause
- [ ] Confirm it does **not** violate an operating rule (single-writer DuckDB, `--profiles-dir .`, never write to `main`, etc.)
- [ ] Apply the change on a branch and re-run `dbt build --profiles-dir .`
- [ ] Open a PR once the build is green

<details><summary>Run metadata</summary>

- `captured_at`: 2026-07-19T13:53:41+00:00
- `dbt_version`: 1.12.0
- `invocation_id`: 6bde060c-3035-4f50-aea6-69e17c717ad3

</details>

---
_This is a **proposal** produced by assisted triage. A human reviews and approves before any change. The pipeline never self-edits._


## Stage 4 - a human applies the fix, pipeline goes green

A human reads the proposed fix, agrees, and restores the correct column. We rebuild to confirm
the loop is closed - back to green.

In [9]:
model.write_text(ORIGINAL)                         # apply the fix (restore the good version)
print("Restored mart_movers.sql.\n")

rc = run(f"{DBT} build --profiles-dir .")
print("\nBUILD:", "GREEN ✅" if rc == 0 else "RED ❌")

# Tidy the triage artifacts we generated (all git-ignored anyway).
for name in ("failure_context.json", "diagnosis.json", "proposed_fix_issue.md"):
    p = PROJECT / name
    if p.exists():
        p.unlink()
print("Cleaned up triage artifacts. The repo is back to its original state.")

Restored mart_movers.sql.

$ /Users/thiha.th/Downloads/market-movers-dbt/.venv/bin/dbt build --profiles-dir .

13:54:12  31 of 33 PASS unique_mart_portfolio_bias_ticker ................................ [PASS in 0.03s]
13:54:12  32 of 33 PASS not_null_mart_momentum_ticker .................................... [PASS in 0.03s]
13:54:12  33 of 33 PASS unique_mart_momentum_ticker ...................................... [PASS in 0.03s]
13:54:12  
13:54:12  Finished running 1 incremental model, 1 seed, 4 table models, 24 data tests, 3 view models in 0 hours 0 minutes and 0.45 seconds (0.45s).
13:54:12  
13:54:12  Completed with 1 error, 0 partial successes, and 0 warnings:
13:54:12  
13:54:12  [ERROR]: in model mart_movers (models/marts/mart_movers.sql)
13:54:12    Runtime Error in model mart_movers (models/marts/mart_movers.sql)
  Binder Error: Values list "r" does not have a column named "close_prices"
  
  LINE 32:     r.close_prices,  -- BUG: injected typo for the self-healing...
          